In [8]:
"""
Weather Prediction Machine Learning Project (Fixed)
"""

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import numpy as np
from pathlib import Path
import pandas as pd


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from sklearn.pipeline import Pipeline



In [9]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

# Clean, single-variable assignment using pathlib
DATA_PATH = Path("mldata-main/weather_prediction_dataset.csv").as_posix()

TARGET_COLUMN = "BASEL_precipitation"
RANDOM_STATE = 42

# Now load the dataset safely
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")



Dataset shape: (3654, 165)


In [10]:
# ============================================================
# 3. LOAD THE DATASET
# ============================================================

print("=" * 70)
print("LOADING DATASET")
print("=" * 70)

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")



LOADING DATASET
Dataset shape: (3654, 165)


In [11]:
# ============================================================
# 4. CHECK TARGET EXISTENCE
# ============================================================

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' not found in dataset.")



In [12]:
# ============================================================
# 5. CONVERT DATE INTO DATETIME FEATURES
# ============================================================

if "DATE" in df.columns:
    print("\nConverting DATE column into time features...")
    df["DATE"] = pd.to_datetime(
        df["DATE"].astype(str),
        format="%Y%m%d",
        errors="coerce"
    )

    df["year"] = df["DATE"].dt.year
    df["month"] = df["DATE"].dt.month
    df["day"] = df["DATE"].dt.day
    df["day_of_week"] = df["DATE"].dt.dayofweek
    df["day_of_year"] = df["DATE"].dt.dayofyear
    df["quarter"] = df["DATE"].dt.quarter
    df["is_weekend"] = (df["DATE"].dt.dayofweek >= 5).astype(int)

    df.drop(columns=["DATE"], inplace=True)




Converting DATE column into time features...


C:\Users\user\AppData\Local\Temp\ipykernel_12896\2262832077.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["year"] = df["DATE"].dt.year
C:\Users\user\AppData\Local\Temp\ipykernel_12896\2262832077.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["month"] = df["DATE"].dt.month
C:\Users\user\AppData\Local\Temp\ipykernel_12896\2262832077.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns

In [13]:
# ============================================================
# 6. SEPARATE FEATURES (X) AND TARGET (y)
# ============================================================

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]


In [14]:
# ============================================================
# 7. DETERMINE PROBLEM TYPE
# ============================================================

if pd.api.types.is_numeric_dtype(y):
    unique_values = y.nunique()
    PROBLEM_TYPE = "classification" if unique_values <= 10 else "regression"
else:
    PROBLEM_TYPE = "classification"

print(f"\nDetected problem type: {PROBLEM_TYPE}")



Detected problem type: regression


In [15]:
# ============================================================
# 8. DROP MISSING TARGETS
# ============================================================

valid_target_rows = y.notna()
X = X.loc[valid_target_rows].copy()
y = y.loc[valid_target_rows].copy()



In [16]:
# ============================================================
# 9. IDENTIFY FEATURE TYPES
# ============================================================

numerical_features = X.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()



In [17]:
# ============================================================
# 10. TRAIN / TEST SPLIT
# ============================================================

if PROBLEM_TYPE == "classification":
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE
    )



In [18]:
# ============================================================
# 11. BUILD PREPROCESSING & MODEL PIPELINE (FIXED)
# ============================================================

# Dynamically assemble transformers to prevent empty list errors
transformers = []

if numerical_features:
    num_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    transformers.append(("numerical", num_pipeline, numerical_features))

if categorical_features:
    cat_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    transformers.append(("categorical", cat_pipeline, categorical_features))

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

# Select Model
if PROBLEM_TYPE == "regression":
    model = RandomForestRegressor(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    )
else:
    model = RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"
    )

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])



In [19]:
# ============================================================
# 12. TRAIN & EVALUATE MODEL
# ============================================================

print("\nTraining Model...")
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print("\n" + "=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)

if PROBLEM_TYPE == "classification":
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1-score:  {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
else:
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R²:   {r2:.4f}")




Training Model...

EVALUATION RESULTS
RMSE: 0.4321
MAE:  0.2146
R²:   0.3599


In [20]:
# ============================================================
# 13. FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("TOP 10 FEATURE IMPORTANCES")
print("=" * 70)

try:
    trained_model = pipeline.named_steps["model"]
    transformed_features = pipeline.named_steps["preprocessor"].get_feature_names_out()
    importance = trained_model.feature_importances_

    feature_importance = pd.DataFrame({
        "Feature": transformed_features,
        "Importance": importance
    }).sort_values(by="Importance", ascending=False)

    print(feature_importance.head(10).to_string(index=False))
except Exception as e:
    print(f"Could not extract feature importances: {e}")


TOP 10 FEATURE IMPORTANCES
                              Feature  Importance
            numerical__BASEL_humidity    0.126053
         numerical__MUENCHEN_pressure    0.103480
    numerical__MUENCHEN_precipitation    0.048071
  numerical__MONTELIMAR_precipitation    0.031310
           numerical__KASSEL_pressure    0.020934
      numerical__KASSEL_precipitation    0.020442
          numerical__DRESDEN_sunshine    0.014421
         numerical__HEATHROW_humidity    0.012602
        numerical__PERPIGNAN_pressure    0.012354
numerical__LJUBLJANA_global_radiation    0.012046


In [21]:
import joblib

# 1. Set up preprocessing for numbers and categories
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 2. Combine preprocessing and the model into a single Pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE))
])

# 3. Train the model
print("Training the model... please wait.")
model_pipeline.fit(X_train, y_train)
print("Model trained successfully!")

# 4. Save the trained pipeline and feature names for deployment
joblib.dump(model_pipeline, 'weather_model.pkl')
joblib.dump(numerical_features, 'numerical_features.pkl')
joblib.dump(categorical_features, 'categorical_features.pkl')
print("Model files saved successfully!")


Training the model... please wait.
Model trained successfully!
Model files saved successfully!
